## **📄 Evaluating AI Medical Note Summarization Applications**

In clinical settings, summarizing medical notes is a critical task that can be significantly improved with AI applications. However, ensuring the **accuracy**, **coherence**, and **safety** of AI-generated summaries is essential before clinical integration. This use case focuses on evaluating AI models for medical note summarization, following the **MedHELM** framework to ensure that outputs meet clinical standards for quality and safety.

## **Medical Summarization Task**

Medical doctors currently spend between **52 to 102 minutes per day** writing clinical notes after patient encounters. These notes document visit discussions, diagnoses, test orders, and treatments. While crucial for healthcare delivery, billing, and legal documentation, this administrative burden—an unintended consequence of the **HITECH Act (2009)**—has become a major contributor to **physician fatigue** and **burnout**. Hiring **medical scribes** has been one solution, but this can cost up to **$112,000 annually** depending on specialty and institution. High attrition rates and specialized training requirements further limit scalability. With the advent of **large language models (LLMs)**, **automatic clinical note generation** is emerging as a promising alternative to reduce physician workload, improve efficiency, and enhance care quality.

However, to responsibly develop and evaluate these AI solutions, **high-quality, representative datasets** are needed. Clinical data is often scarce and highly protected due to ethical and legal concerns. For this project, we will use the **ACI-Bench** dataset — a synthetic yet clinically realistic corpus of full visit dialogues and corresponding clinical notes — to benchmark and evaluate model performance.

> **Note**: Although the broader corpus also includes the **MTS-Dialog** dataset, **we will only use ACI-Bench** for our evaluation.  
> ACI-Bench is more aligned with our goals because it contains **full visit dialogues**, **comprehensive clinical notes**, and **rich metadata** (e.g., patient age, gender), offering a closer simulation of real-world clinical encounters.

### **⚙️ What We’ll Build**

An end‑to‑end medical note‑summarization application that tests multiple large language models (LLMs) and systematically benchmarks their ability to generate concise, clinically accurate visit notes.

**Key Steps**  
1. **Model Selection** — compare GPT‑4o, O1, Phi‑3, Phi‑4, and other state‑of‑the‑art models.  
2. **Find Dataset** — map summarization tasks to established clinical taxonomies and Datasets.  
3. **Evaluation Methods** — blend rule‑based validation with LLM‑as‑Judge assessments.  
5. **Leaderboard & Visualization** — surface results in an interactive dashboard (Azure AI Foundry or a custom Streamlit app).

**Why It Matters**  
A rigorous, transparent evaluation pipeline protects patient safety, lightens clinicians’ documentation burden, and accelerates the responsible adoption of AI in healthcare.

### **📚 Dataset Overview**

- **ACI-Bench**:  
  - Full visit dialogues between clinician and patient.  
  - Complete clinical notes paired with the dialogues.  
  - Metadata such as patient age, gender, and name.  
  - Synthetic data curated by medical annotators and clinicians.

The **ACI-Bench** dataset has been used in major international research challenges, including:

- [MEDIQA-CHAT 2023 @ ACL CLINICALNLP](https://sites.google.com/view/mediqa2023/clinicalnlp-mediqa-chat-2023)
- [MEDIQA-SUM 2023 @ CLEF](https://www.imageclef.org/2023/medical/mediqa)

This dataset enables responsible experimentation and benchmarking for AI models focused on clinical note generation.

### **📝 Papers**

**ACI-BENCH: A Novel Ambient Clinical Intelligence Dataset for Benchmarking Automatic Visit Note Generation**  
Authors: Wen-wai Yim, Yujuan Fu, Asma Ben Abacha, Neal Snider, Thomas Lin, Meliha Yetisgen  
[Paper Link](https://www.nature.com/articles/s41597-023-02487-3)

```bibtex
@article{aci-bench,
  author  = {Wen-wai Yim and Yujuan Fu and Asma Ben Abacha and Neal Snider and Thomas Lin and Meliha Yetisgen},
  title   = {ACI-BENCH: a Novel Ambient Clinical Intelligence Dataset for Benchmarking Automatic Visit Note Generation},
  journal = {Nature Scientific Data},
  year    = {2023},
  volume  = {10},
  url     = {https://www.nature.com/articles/s41597-023-02487-3}
}
```


## **1. Choosing the Candidate LLMs**

As I navigated the Azure AI Foundry catalog, a plethora of AI models unfolded, each with unique capabilities. My past successes with Azure OpenAI models naturally directed my focus towards them for my current endeavor: offline medical note summarization.​

Given the offline nature of this application, real-time processing isn't a necessity. Therefore, my primary concern shifts from latency to the quality of the output. Cost-effectiveness remains a consideration, but the accuracy and coherence of the generated summaries are paramount.​


<br>

<div align="center">
  <img src="../utils/images/AICatalog.png" alt="Solution Diagram" width="80%" />
</div>

<br>

With these priorities in mind, I've shortlisted the following models for evaluation:​

+ **GPT-4o**: Renowned for its high-quality outputs, GPT-4o offers advanced capabilities, including multimodal processing. However, these features come with higher costs and latency, which might be excessive for an offline application.​

- **GPT-4o Mini**: A streamlined version of GPT-4o, this model balances performance and cost. It boasts a 128K token context window and has demonstrated superior performance on benchmarks like MMLU, scoring 82% compared to GPT-3.5 Turbo's 70%. Additionally, it's more than 60% cost-effective than GPT-3.5 Turbo, making it a compelling choice for applications where quality and cost are both crucial considerations. ​

- **GPT-4.1 Nano**: The latest addition to the GPT series, GPT-4.1 Nano is designed for low latency and cost. It supports a context window of up to 1 million tokens, significantly surpassing its predecessors. This model is tailored for applications requiring efficient processing of large datasets, making it suitable for offline medical note summarization.​

+ **o4-mini**: This model is optimized for reasoning tasks, such as mathematical problem-solving and logical reasoning. Including o4-mini in the evaluation allows for a comparison between instruction-following models and those designed for reasoning, providing insights into their respective performances in medical note summarization. ​
GeeksforGeeks


By evaluating these models, I aim to identify the one that offers the best balance between quality and cost for my specific use case. The advancements in GPT-4.1 Nano, particularly its expansive context window and cost-efficiency, make it a promising candidate for applications like mine.

In [39]:
import os
import time
from typing import List, Dict, Tuple
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

import logging
logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)

# We begin our journey by loading the secret keys from the environment...
ENDPOINT: str = "https://aihubnw3478841489.openai.azure.com/"
API_VERSION: str = "2024-12-01-preview"
SUBSCRIPTION_KEY: str = os.getenv("AZURE_OPENAI_KEY") or ""

if not SUBSCRIPTION_KEY:
    raise ValueError("Please set the AZURE_OPENAI_KEY environment variable.")

# Here we map conversational aliases to actual Azure deployment identifiers.
MODEL_CONFIGS: Dict[str, str] = {
    "gpt-4o": "gpt-4o",
    "gpt-4o-mini": "gpt-4o-mini",
    "gpt-4.1-nano": "gpt-4.1-nano",
    "o4-mini": "o4-mini", 
}

# Client Initialization 
def get_openai_client() -> AzureOpenAI:
    """
    Initialize and return an Azure OpenAI client.
    """
    logger.info("Initializing Azure OpenAI client.")
    return AzureOpenAI(
        api_version=API_VERSION,
        azure_endpoint=ENDPOINT,
        api_key=SUBSCRIPTION_KEY,
    )


# Instruction-following Models
def call_chat_model(
    client: AzureOpenAI,
    model_key: str,
    messages: List[Dict[str, str]],
    max_tokens: int = 1024,
    temperature: float = 1.0,
    top_p: float = 1.0,
) -> Tuple[str, float]:
    """
    Call instruction-following models and measure end-to-end latency.
    Returns:
      - response text
      - latency in seconds
    """
    deployment = MODEL_CONFIGS[model_key]
    logger.info("Calling chat model '%s'...", deployment)
    start = time.time()

    try:
        response = client.chat.completions.create(
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            model=deployment,
        )
    except Exception as e:
        logger.exception("Chat model call failed for model '%s': %s", deployment, e)
        raise

    latency = time.time() - start
    logger.info("Model '%s' responded in %.2fs", deployment, latency)
    return response.choices[0].message.content, latency


# Reasoning-optimized Models 
def call_reasoning_model(
    client: AzureOpenAI,
    model_key: str,
    messages: List[Dict[str, str]],
    max_completion_tokens: int = 100000,
) -> Tuple[str, float]:
    """
    Call reasoning-optimized models (use max_completion_tokens) and measure latency.
    Returns:
      - response text
      - latency in seconds
    """
    deployment = MODEL_CONFIGS[model_key]
    logger.info("Calling reasoning model '%s'...", deployment)
    start = time.time()

    try:
        response = client.chat.completions.create(
            messages=messages,
            max_completion_tokens=max_completion_tokens,
            model=deployment,
        )
    except Exception as e:
        logger.exception("Reasoning model call failed for model '%s': %s", deployment, e)
        raise

    latency = time.time() - start
    logger.info("Model '%s' responded in %.2fs", deployment, latency)
    return response.choices[0].message.content, latency


client: AzureOpenAI = get_openai_client()

test_messages: List[Dict[str, str]] = [
    {
        "role": "system",
        "content": (
            "You are a medical scribe assistant. "
            "The following is a very messy, run-on clinical consultation note. "
            "Please clarify the key medical findings and then summarize it into a clean SOAP note (Subjective, Objective, Assessment, Plan)."
        )
    },
    {
        "role": "user",
        "content": (
            "John Doe a 45 year old male with hx of hypertension and hyperlipidemia presents complaining of chest pressure "
            "and shortness of breath for the last three days episodes happen with climbing stairs lasting about 10 to 15 mins "
            "eased by rest and nitroglycerin one dose also noticed mild sweating no nausea vital signs today BP 150/92 HR 88 "
            "RR 18 T 98.6 SpO2 97% physical exam heart regular no murmurs chest wall tender lungs clear pulses equal labs "
            "ECG normal sinus no ST changes troponin slightly elevated 0.04 plan to get stress test continue aspirin statin "
            "increase lisinopril advise diet exercise follow up in one week"
        )
    }
]

# Testing our instruction-following model
chat_content, chat_latency = call_chat_model(
    client, "gpt-4o", test_messages, max_tokens=1000, temperature=0.7, top_p=1.0
)
print(f"GPT-4o says: {chat_content}")
print(f"GPT-4o latency: {chat_latency:.3f}s\n")

# Testing our reasoning model
reason_content, reason_latency = call_reasoning_model(
    client, "o4-mini", test_messages, max_completion_tokens=100000
)
print(f"O4-mini reasons: {reason_content}")
print(f"O4-mini latency: {reason_latency:.3f}s")


2025-04-28 00:03:07,577 [INFO] Initializing Azure OpenAI client.


2025-04-28 00:03:07,632 [INFO] Calling chat model 'gpt-4o'...
2025-04-28 00:03:12,049 [INFO] HTTP Request: POST https://aihubnw3478841489.openai.azure.com//openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2025-04-28 00:03:12,053 [INFO] Model 'gpt-4o' responded in 4.42s
2025-04-28 00:03:12,057 [INFO] Calling reasoning model 'o4-mini'...


GPT-4o says: ### SOAP Note

**Patient Name:** John Doe  
**Age/Gender:** 45-year-old male  
**Medical History:** Hypertension, Hyperlipidemia  

---

**Subjective:**  
John Doe reports 3 days of intermittent chest pressure and shortness of breath, occurring during exertion such as climbing stairs. Each episode lasts approximately 10-15 minutes and is relieved by rest and one dose of nitroglycerin. He also noted mild sweating during these episodes but denies nausea.  

---

**Objective:**  
- **Vital Signs:** BP 150/92, HR 88, RR 18, Temp 98.6°F, SpO2 97%  
- **Physical Exam:**  
  - Heart: Regular rate and rhythm, no murmurs  
  - Lungs: Clear bilaterally  
  - Chest wall: Tenderness noted  
  - Pulses: Equal bilaterally  
- **Labs & Imaging:**  
  - ECG: Normal sinus rhythm, no ST changes  
  - Troponin: Slightly elevated at 0.04  

---

**Assessment:**  
1. Chest pain with exertion concerning for possible angina.  
2. Elevated troponin suggests possible myocardial ischemia.  
3. Hype

2025-04-28 00:03:29,199 [INFO] HTTP Request: POST https://aihubnw3478841489.openai.azure.com//openai/deployments/o4-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2025-04-28 00:03:29,204 [INFO] Model 'o4-mini' responded in 17.15s


O4-mini reasons: Key Medical Findings  
• 45‑year‑old male with known hypertension and hyperlipidemia  
• 3‑day history of substernal chest pressure and shortness of breath on exertion (e.g., climbing stairs), lasting 10–15 minutes and relieved by rest and nitroglycerin  
• Associated mild diaphoresis, no nausea or vomiting  
• Vital signs: BP 150/92 mmHg, HR 88 bpm, RR 18, T 98.6°F, SpO₂ 97% on room air  
• Cardiovascular exam: regular rate and rhythm, no murmurs, no chest wall tenderness  
• Pulmonary exam: clear to auscultation bilaterally  
• Peripheral pulses equal and symmetrical  
• ECG: normal sinus rhythm, no ST‑T wave changes suggestive of acute ischemia  
• Cardiac biomarkers: troponin I mildly elevated at 0.04 ng/mL  

SOAP Note  

Subjective  
- Patient: John Doe, 45‑year‑old male  
- Chief complaint: “Pressure in my chest and shortness of breath when I climb stairs” for the past 3 days  
- History of present illness: Episodic substernal chest pressure lasting ~10–15 minut

## **2. Find The Ground Truth (Dataset) — My “Reason-Backwards” Process**
> **Why ACI‑Bench and *not* dataset X?**  
> My answer follows MedHELM’s pillars of **(a) real‑world relevance, (b) multi‑metric rigor, (c) standardized replicability**.  
> Below is the exact decision pipeline— I followed.

#### **2.1 Start with the Task Taxonomy → then Search for Data**
1. **Enumerate real tasks.** MedHELM crowdsourced 121 clinician‑approved tasks across five categories.  
2. **Pick one high‑impact sub‑task.** *Clinical Note Generation* drains the most clinician time; automating it would reclaim entire clinic days.  
3. **Define the evaluation unit.** One visit transcript ➔ one complete SOAP note. Any benchmark must test that end‑to‑end behavior.

#### **2.2 Research Public Datasets Available**

**Auditor:** “Which corpora even qualify for end‑to‑end note generation?”  
**Me:** “We need (1) full visit dialogue, (2) a gold clinical note, (3) ≥ 200 cases, (4) public access, and (5) SOAP‑compatible formatting.”  

| Dataset        | Full dialogue + note | Public licence | ≥ 200 cases | SOAP‑ready | Verdict |
|----------------|----------------------|----------------|-------------|------------|---------|
| MTS‑Dialog     | ✗ (snippets only)    | ✓              | 1.7 k       | ✗          | ❌ Reject |
| Primock‑57     | ✓                    | ✓              | 57          | ✗          | ❌ Too small |
| MIMIC notes    | ✗ (no dialogue)      | Restricted     | 38 k        | n/a        | ❌ No dialogue |
| Private EHR    | ✓                    | ✗              | ✓           | varies     | ❌ PHI‑restricted |
| **ACI‑Bench**  | **✓**                | **CC‑BY‑4.0**  | **207**     | **✓**      | ✅ **Select** |

**Auditor:** “So ACI‑Bench is literally the only publicly available corpus that ticks every box.”  
**Me:** “Absolutely. ACI‑Bench not only meets every benchmark but exceeds our expectations—saving us the time and complexity of building a private dataset from scratch.”

#### **2.3 Map Task → Dataset → Benchmark Artefacts**

For the first experiment I’m sticking to a strictly zero-shot setup: one realistic instruction, no section-by-section prompts. Prompt engineering is a fascinating optimisation topic, but to establish a clean baseline I want every model to process exactly the same input.

+ **Single Prompt** – the ClinScribe-AI instruction shown below.
- **Models Under Test** – gpt-4o, gpt-4o-mini, gpt-4.1-nano, and the reasoning-tuned o3-mini.
+ **Run** – feed each ACI-Bench transcript once and capture the full SOAP note.
- **Later Layers** – after we see out-of-the-box behaviour, we’ll add evaluation metrics, prompt-tuning, and chain-of-thought comparisons.

This baseline tells us which model already “gets it right” before any prompt hacks—measuring genuine capability rather than prompt-engineering skill.

**SYSTEM**  
You are *ClinScribe‑AI*, a licensed medical scribe with 10 years of primary‑care experience.  
• Draft concise, clinically accurate notes for the attending physician.  
• If facts are missing, write “—” instead of hallucinating.  
• NEVER invent test results, patient identifiers, or medication doses.

**INSTRUCTIONS**  
1. Identify key findings → map to SOAP sections.  
2. Verify no unsupported data.  
3. Assemble final note exactly as instructed.

**USER**  
Summarize the following **doctor–patient conversation** into one clinical note with **exactly four section headers** (ALL‑CAPS, followed by a colon) in this order:

HISTORY OF PRESENT ILLNESS:  
PHYSICAL EXAM:  
RESULTS:  
ASSESSMENT AND PLAN:  

Insert a blank line *after* each header.  
Return nothing else—no explanations, no markdown.  

Conversation (delimited by ```):  
{{transcript_text}}

In [107]:
"""
This script demonstrates how to call both instruction-following and reasoning-optimized models using the Azure OpenAI API.
Run single-shot SOAP-note generation on a slice of ACI-Bench.
"""
import pandas as pd

# config 
MODEL_CONFIGS: Dict[str, str] = {
    "gpt-4o":       "gpt-4o",
    "gpt-4o-mini":  "gpt-4o-mini",
    "gpt-4.1-nano": "gpt-4.1-nano",
    "o3-mini":      "o3-mini",
}
REASONING_MODELS = {"o3-mini"}

MAX_TOKENS     = 5000        # generous for full-note
TEMPERATURE    = 0.7
TOP_P          = 1.0
N_EXAMPLES     = 1         # how many rows of df to test

# ── prompt builder
SYSTEM_PROMPT = """\
You are ClinScribe-AI, a licensed medical scribe with 10 years of primary-care experience.
• Draft concise, clinically accurate notes for the attending physician.
• If facts are missing, write "—" instead of hallucinating.
• NEVER invent test results, patient identifiers, or medication doses.

INSTRUCTIONS:
1. Identify key findings → map to SOAP sections.
2. Verify no unsupported data.
3. Assemble final note exactly as instructed.
"""

USER_PROMPT = """\
Summarize the following doctor–patient conversation into one clinical note with **exactly four section headers** (ALL-CAPS, followed by a colon) in this order:

HISTORY OF PRESENT ILLNESS:
PHYSICAL EXAM:
RESULTS:
ASSESSMENT AND PLAN:

Insert a blank line after each header.
Return nothing else—no explanations, no markdown.

Conversation (delimited by ```):
{dialogue}
"""

def build_messages(dialogue: str) -> List[Dict[str, str]]:
    return [{"role": "user", "content": USER_PROMPT.format(dialogue=dialogue)}]

df_evals = pd.read_csv("data/ground_truth.csv").head(N_EXAMPLES)


In [108]:
from openai import AsyncAzureOpenAI
import asyncio

SUBSCRIPTION_KEY: str = os.getenv("AZURE_OPENAI_KEY") or ""

if not SUBSCRIPTION_KEY:
    raise ValueError("Please set the AZURE_OPENAI_KEY environment variable.")

async_client = AsyncAzureOpenAI(
    api_key=SUBSCRIPTION_KEY,
    azure_endpoint=ENDPOINT,
    api_version=API_VERSION,
)

async def call_chat_model_async(
    deployment: str,
    messages: List[Dict[str, str]],
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
    max_tokens: int = MAX_TOKENS,
) -> Tuple[str, float]:
    """
    Async call to instruction‑following (chat) models with full
    error handling and latency tracking.
    """
    logger.info("Submitting async chat request to '%s'...", deployment)
    start = time.time()
    try:
        resp = await async_client.chat.completions.create(
            model=deployment,
            messages=messages,
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
        )
        latency = time.time() - start
        logger.info("Chat model '%s' responded in %.2f s", deployment, latency)
        return resp.choices[0].message.content, latency
    except Exception as e:
        latency = time.time() - start
        logger.exception(
            "Chat model '%s' failed after %.2f s: %s",
            deployment,
            latency,
            e,
        )
        raise


async def call_reasoning_model_async(
    deployment: str,
    messages: List[Dict[str, str]],
    max_completion_tokens: int = MAX_TOKENS,
) -> Tuple[str, float]:
    """
    Async call to reasoning‑optimized models with full
    error handling and latency tracking.
    """
    logger.info("Submitting async reasoning request to '%s'...", deployment)
    start = time.time()
    try:
        resp = await async_client.chat.completions.create(
            model=deployment,
            messages=messages,
            max_completion_tokens=max_completion_tokens,
        )
        latency = time.time() - start
        logger.info("Reasoning model '%s' responded in %.2f s", deployment, latency)
        return resp.choices[0].message.content, latency
    except Exception as e:
        latency = time.time() - start
        logger.exception(
            "Reasoning model '%s' failed after %.2f s: %s",
            deployment,
            latency,
            e,
        )
        raise


async def process_encounter(enc, models: Dict[str, str], records: List[Dict]):
    """
    For one encounter, fire off all model calls in parallel and
    record results.  Any individual failure is logged but does
    not abort the rest.
    """
    try:
        messages = build_messages(enc.dialogue)
    except Exception as e:
        logger.exception("Failed to build messages for encounter %s: %s", enc.encounter_id, e)
        return records

    tasks, keys = [], []
    for key, dep in models.items():
        keys.append(key)
        coro = (
            call_reasoning_model_async(dep, messages, 10000)
            if key in REASONING_MODELS
            else call_chat_model_async(dep, messages)
        )
        tasks.append(asyncio.create_task(coro))

    results = await asyncio.gather(*tasks, return_exceptions=True)

    for key, result in zip(keys, results):
        if isinstance(result, Exception):
            logger.error("Model %s failed on encounter %s: %s", key, enc.encounter_id, result)
            continue

        note, latency = result
        records.append(
            {
                "encounter_id": enc.encounter_id,
                "model": key,
                "latency_sec": latency,
                "response": note,
                "ground_truth": enc.note,
            }
        )
        logger.info("Completed %s for encounter %s in %.2f s", key, enc.encounter_id, latency)

    return records


In [109]:
# Load only the first 10 encounters
df = pd.read_csv("data/ground_truth.csv").head(10)

records = []
# Build and run only the first 10 in parallel
tasks = [ process_encounter(row, MODEL_CONFIGS, records) for row in df.itertuples() ]
records = await asyncio.gather(*tasks)


2025-04-28 01:11:40,272 [INFO] Submitting async chat request to 'gpt-4o'...
2025-04-28 01:11:40,281 [INFO] Submitting async chat request to 'gpt-4o-mini'...


2025-04-28 01:11:40,289 [INFO] Submitting async chat request to 'gpt-4.1-nano'...
2025-04-28 01:11:40,295 [INFO] Submitting async reasoning request to 'o3-mini'...
2025-04-28 01:11:40,302 [INFO] Submitting async chat request to 'gpt-4o'...
2025-04-28 01:11:40,309 [INFO] Submitting async chat request to 'gpt-4o-mini'...
2025-04-28 01:11:40,314 [INFO] Submitting async chat request to 'gpt-4.1-nano'...
2025-04-28 01:11:40,321 [INFO] Submitting async reasoning request to 'o3-mini'...
2025-04-28 01:11:40,325 [INFO] Submitting async chat request to 'gpt-4o'...
2025-04-28 01:11:40,330 [INFO] Submitting async chat request to 'gpt-4o-mini'...
2025-04-28 01:11:40,338 [INFO] Submitting async chat request to 'gpt-4.1-nano'...
2025-04-28 01:11:40,342 [INFO] Submitting async reasoning request to 'o3-mini'...
2025-04-28 01:11:40,349 [INFO] Submitting async chat request to 'gpt-4o'...
2025-04-28 01:11:40,356 [INFO] Submitting async chat request to 'gpt-4o-mini'...
2025-04-28 01:11:40,362 [INFO] Submit

In [ ]:
import pandas as pd

if any(isinstance(r, list) for r in records):
    flat_records = [r for batch in records for r in batch]
else:
    flat_records = records


df = pd.DataFrame(flat_records)
df = df.drop_duplicates(subset=['model', 'encounter_id'])
df = df[['model','encounter_id','response','latency_sec','ground_truth']]
df = df.set_index(['model','encounter_id'])
by_model = {
    model: grp.reset_index(drop=True)
    for model, grp in df.reset_index().groupby('model')
}

by_model['gpt-4.1-nano'].head(5)


,model,encounter_id,response,latency_sec,ground_truth
0,gpt-4.1-nano,D2N009,"HISTORY OF PRESENT ILLNESS:\nBryan, a 55-year-...",2.066208,CHIEF COMPLAINT\n\nBack pain.\n\nHISTORY OF PR...
1,gpt-4.1-nano,D2N007,"HISTORY OF PRESENT ILLNESS:\nSarah, a 27-year-...",2.270270,CHIEF COMPLAINT\n\nAnnual visit.\n\nHISTORY OF...
2,gpt-4.1-nano,D2N008,"HISTORY OF PRESENT ILLNESS: Stephanie, a 49-ye...",2.578084,CHIEF COMPLAINT\n\nAbnormal labs.\n\nHISTORY O...
3,gpt-4.1-nano,D2N010,HISTORY OF PRESENT ILLNESS:\nKeith is a 58-yea...,2.475677,CHIEF COMPLAINT\n\nHigh blood sugar.\n\nHISTOR...
4,gpt-4.1-nano,D2N004,"HISTORY OF PRESENT ILLNESS:\nJames, a 57-year-...",2.795061,CHIEF COMPLAINT\n\nBack pain.\n\nHISTORY OF PR...
5,gpt-4.1-nano,D2N001,HISTORY OF PRESENT ILLNESS: \nMartha is a 50-...,2.721537,CHIEF COMPLAINT\n\nAnnual exam.\n\nHISTORY OF ...
6,gpt-4.1-nano,D2N006,HISTORY OF PRESENT ILLNESS:\nAnna is a 44-year...,2.922215,CHIEF COMPLAINT\n\nFollow-up of chronic proble...
7,gpt-4.1-nano,D2N003,HISTORY OF PRESENT ILLNESS:\nJohn is a 61-year...,2.234447,CHIEF COMPLAINT\n\nBack pain.\n\nHISTORY OF PR...
8,gpt-4.1-nano,D2N005,"HISTORY OF PRESENT ILLNESS:\nMs. Hill, a 41-ye...",1.794460,CC:\n\nRight middle finger pain.\n\nHPI:\n\nMs...
9,gpt-4.1-nano,D2N002,HISTORY OF PRESENT ILLNESS: \nAndrew is a 62-...,2.311966,CHIEF COMPLAINT\n\nJoint pain.\n\nHISTORY OF P...


## **3. Choosing the Evaluation Metrics**

To assess how well each model captures the critical clinical facts, we focus on **Relevance** and combine a **deterministic automated score** with an **LLM-as-judge** layer.

### 3.1 Deterministic Automated Relevance

We compute two complementary metrics and average them:

1. **Lexical Overlap (ROUGE-1/2/L)**  
   - **Why?** Quickly checks n-gram recall/precision for key terms (e.g., “chest pressure”, “nitroglycerin”).  
   - **How:**  
     ```python
     rouge1 = rouge_f1(candidate, reference, n=1)
     rouge2 = rouge_f1(candidate, reference, n=2)
     rougel = rouge_l_f1(candidate, reference)
     ROUGE = (rouge1 + rouge2 + rougel) / 3
     ```

2. **Clinical Concept F1 (MedCon)**  
   - **Why?** Verifies no hallucinated or missing medical entities (drugs, disorders, anatomy).  
   - **How:**  
     1. Extract UMLS concepts with QuickUMLS from both candidate and reference, restricted to semantic groups:
        - Anatomy  
        - Chemicals & Drugs  
        - Disorders  
        - Phenomena & Physiology  
     2. Compute strict set‐F1 between the two concept sets:
     ```python
     medcon_f1 = f1_score(set(quickumls(candidate)), set(quickumls(reference)))
     ```

3. **Deterministic Relevance**  
   - **Formula:**  
     ```
     deterministic_relevance = (ROUGE + medcon_f1) / 2
     ```

### 3.2 LLM-as-Judge Relevance

Automated metrics can miss subtle clinical or stylistic issues. We therefore ask an LLM (e.g. GPT-4o) to rate each note:

> **Prompt:**  
> ```
> Here are the REFERENCE and MODEL-GENERATED SOAP notes.
> Rate the MODEL note on a 1–5 scale for:
>  • Completeness: captured every key finding?  
>  • Correctness: no invented or contradictory facts?  
>  • Clarity: clear, concise, professional style?  
>  • Overall Relevance.
>
> Return JSON: {"completeness": X, "correctness": Y, "clarity": Z, "overall": W}
> ```

We then **average** the `"overall"` scores across all encounters to get the **LLM-judged Relevance**.

### 3.3 Composite Scoring

| Score Type                   | Calculation                                    | Range           |
|-------------------------------|------------------------------------------------|-----------------|
| **Deterministic Relevance**   | `(ROUGE + MedCon) / 2`                         | 0.0 – 1.0       |
| **LLM-Judged Relevance**      | Mean of `"overall"` from JSON rubric           | 1 – 5           |

**Combining these two** gives us:
- A **reproducible backbone** (fast, numeric, CI-friendly).  
- A **human-like clinical check** that catches nuance and safety concerns.  


In [ ]:
import os
import time
import json
from dataclasses import dataclass
from typing import List, Dict

import pandas as pd
from rapidfuzz import fuzz
from quickumls import QuickUMLS

from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from openai import AzureOpenAI
from azure.ai.projects.models import (
    Evaluation, Dataset, EvaluatorConfiguration, ConnectionType
)
from azure.ai.evaluation import F1ScoreEvaluator, RelevanceEvaluator, ViolenceEvaluator


AZ_ENDPOINT = "https://aihubnw3478841489.openai.azure.com/"
AZ_API_KEY  = os.getenv("AZURE_OPENAI_KEY")
if not AZ_API_KEY:
    raise ValueError("Please set AZURE_OPENAI_KEY in your environment")

# For LLM-as-Judge
llm_client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint=AZ_ENDPOINT,
    api_key=AZ_API_KEY
)

# For evaluation pipeline
AZURE_AI_FOUNDRY_CONNECTION_STRING = os.getenv("AZURE_AI_FOUNDRY_CONNECTION_STRING")

credential = DefaultAzureCredential()

ai_project_client = AIProjectClient.from_connection_string(
    credential=DefaultAzureCredential(),
    conn_str=AZURE_AI_FOUNDRY_CONNECTION_STRING,
)


@dataclass
class RougeScore:
    rouge: float

class RougeEvaluator:
    def __call__(self, candidate: str, reference: str) -> RougeScore:
        # simple ROUGE-1/2/L with RapidFuzz’s ratio as proxy
        r1 = fuzz.partial_ratio(reference, candidate) / 100
        r2 = fuzz.token_set_ratio(reference, candidate) / 100
        rl = fuzz.QRatio(reference, candidate) / 100
        return RougeScore(rouge=(r1 + r2 + rl) / 3)

@dataclass
class MedConScore:
    medcon_f1: float

class MedConEvaluator:
    def __init__(self):
        # point to your UMLS installation
        self.umls = QuickUMLS('/path/to/quickumls', dokany=False)

    def __call__(self, candidate: str, reference: str) -> MedConScore:
        c_concepts = {c[0][0] for c in self.umls.extract(candidate)}
        r_concepts = {c[0][0] for c in self.umls.extract(reference)}
        tp = len(c_concepts & r_concepts)
        p  = len(c_concepts) or 1
        r  = len(r_concepts) or 1
        f1 = 2 * tp / (p + r)
        return MedConScore(medcon_f1=f1)

@dataclass
class LLMJudgeScore:
    completeness: int
    correctness:   int
    clarity:       int
    overall:       int

class LLMJudgeEvaluator:
    PROMPT = """
Here are the REFERENCE and MODEL-GENERATED SOAP notes.
Rate the MODEL note on a 1–5 scale for:
 • Completeness: captured every key finding?
 • Correctness: no invented or contradictory facts?
 • Clarity: clear, concise, professional style?
 • Overall Relevance.

Return exactly a JSON object like:
{{"completeness": X, "correctness": Y, "clarity": Z, "overall": W}}
"""

    def __init__(self, client: AzureOpenAI):
        self.client = client

    def __call__(self, candidate: str, reference: str) -> LLMJudgeScore:
        msg = [
            {"role": "system", "content": "You are a medical note evaluator."},
            {"role": "user",   "content": self.PROMPT +
                "\n\nREFERENCE:\n" + reference +
                "\n\nMODEL:\n"     + candidate
            }
        ]
        resp = self.client.chat.completions.create(
            model="gpt-4o",
            messages=msg,
            temperature=0.0,
            max_completion_tokens=200
        )
        data = json.loads(resp.choices[0].message.content)
        return LLMJudgeScore(**data)


In [ ]:
from azure.ai.evaluation import evaluate, ContentSafetyEvaluator

# We'll run evaluate(...) with these evaluators.
results = evaluate(
    evaluation_name="healthcare-eval",
    data=str(eval_data_path),
    evaluators={
        "f1_score": f1_eval,
        "relevance": rel_eval,
        "groundedness": ground_eval,
    },
    azure_ai_project=None, #local
    evaluator_config={
        "f1_score": {
            "column_mapping": {
                "response": "${data.response}",
                "ground_truth": "${data.ground_truth}"
            }
        },
        "relevance": {
            "column_mapping": {
                "query": "${data.query}",
                "response": "${data.response}"
            }
        },
        "groundedness": {
            "column_mapping": {
                "context": "${data.context}",
                "response": "${data.response}"
            }
        }
    }
)

## **4. Generate Leaderboard**
